# Housing Workflow

Coursework study extract, week 2.

Inspect housing data, build train-only preprocessing pipelines, compare a baseline, and tune a random forest with cross-validation. First execution may download California housing through scikit-learn. No dataset or fitted model is bundled.

Adapted from completed code in `ML_CSAI500_M2_In_Class_Activity.ipynb`. Instructor-provided scaffolding and pair-work context are acknowledged; authorship of individual lines cannot be independently separated. See [ATTRIBUTION.md](../ATTRIBUTION.md). Teaching prompts, supplied solutions, saved outputs, and identifying metadata are omitted.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
pd.set_option('display.max_columns',20)
RANDOM_STATE=42


In [ ]:
housing=fetch_california_housing(as_frame=True)
df=housing.frame.copy()
print('Shape:',df.shape)
display(df.head())


In [ ]:
quality_demo=pd.DataFrame({'Age':[22,35,np.nan,41,190],'Income':[42000,68000,52000,np.nan,71000],'Plan':['Basic','Premium','Basic','Premium','Unknown']})
quality_demo.info()
display(quality_demo.describe().T)

print('Missing values:')
display(quality_demo.isna().sum().sort_values(ascending=False).head())



In [ ]:
actual=[1,1,1,1,0,0,0,0]
predicted=[1,1,0,1,1,0,0,0]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

print('Accuracy :', accuracy_score(actual, predicted))
print('Precision:', precision_score(actual, predicted))
print('Recall   :', recall_score(actual, predicted))
print('F1 score :', f1_score(actual, predicted))
print()

cm = confusion_matrix(actual,predicted)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[0,1]
).plot(cmap="Blues")

print()


In [ ]:
category_demo=pd.DataFrame({'Income':[25000,50000,90000,70000],'Color':['Red','Blue','Green','Purple']})
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded_colors = encoder.fit_transform(category_demo[["Color"]])

encoded = pd.DataFrame(encoded_colors, columns=encoder.get_feature_names_out(["Color"]))
display(pd.concat([category_demo[["Income"]], encoded], axis=1))


normalized_income = MinMaxScaler().fit_transform(category_demo[["Income"]])
print("Normalized income:", normalized_income.ravel().round(2))

standardized_income = StandardScaler().fit_transform(category_demo[["Income"]])
print("Standardized income:", standardized_income.ravel().round(2))

print()


In [ ]:
print('Shape:', df.shape)


display(df.head())
display(df.info())
display(df.describe().T)
display(df.isna().sum())


In [ ]:
import pandas as pd

features = df.drop(
    columns=["MedHouseVal", "Latitude", "Longitude"]
)

Q1 = features.quantile(0.25)
Q3 = features.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outlier_mask = (features < lower) | (features > upper)

outlier_summary = pd.DataFrame({
    "Outliers": outlier_mask.sum(),
    "Percentage": outlier_mask.mean() * 100,
    "Lower Bound": lower,
    "Upper Bound": upper
})

display(outlier_summary.round(2))


In [ ]:
df.hist(figsize=(12,8),bins=30)
plt.suptitle('Feature and target distributions')
plt.tight_layout()
plt.show()


In [ ]:
corr=df.corr(numeric_only=True)
display(corr['MedHouseVal'].sort_values(ascending=False).to_frame('correlation with target'))
plt.figure(figsize=(10,7))
plt.imshow(corr,cmap='coolwarm',vmin=-1,vmax=1)
plt.colorbar(label='correlation')
plt.xticks(range(len(corr.columns)),corr.columns,rotation=90)
plt.yticks(range(len(corr.columns)),corr.columns)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()


In [ ]:
X = df.drop(columns = 'MedHouseVal')
y = df['MedHouseVal']
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

display(
    X_train.shape,
    X_test.shape,
    y_train.shape,
    y_test.shape
)


In [ ]:
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy = 'median')), ('scaler', StandardScaler())])

preprocessor = ColumnTransformer([('numeric', numeric_pipeline, list(X.columns))])

X_train_prepared = preprocessor.fit_transform(X_train)

X_test_prepared = preprocessor.transform(X_test)

print('Prepared shapes:',X_train_prepared.shape,X_test_prepared.shape)


In [ ]:
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train,y_train)
baseline_rmse=mean_squared_error(y_train,baseline.predict(X_train))**0.5
linear_pipeline=Pipeline([('preprocessor',preprocessor),('model',LinearRegression())])
linear_pipeline.fit(X_train,y_train)
linear_rmse=mean_squared_error(y_train,linear_pipeline.predict(X_train))**0.5

print('Baseline RMSE:',round(baseline_rmse,3))
print('Linear RMSE:',round(linear_rmse,3))


In [ ]:
rmse_scores = -cross_val_score(
    linear_pipeline,
    X_train,
    y_train,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=1
)
print('Fold RMSE:', rmse_scores.round(3))
print('Mean:', round(rmse_scores.mean(), 3))
print('Std:', round(rmse_scores.std(), 3))


In [ ]:
forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators': [30, 60],
    'model__max_features': [0.7, 1.0]
}

grid_search = GridSearchCV(
    forest_pipeline,
    param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=1
)
grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV RMSE:', round(-grid_search.best_score_, 3))


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

final_model = grid_search.best_estimator_

predictions = final_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print('Final Test RMSE:', round(rmse, 3))
print('Final Test MAE:', round(mae, 3))
print('Final Test R²:', round(r2, 3))

results = pd.DataFrame({
    'Actual': np.asarray(y_test),
    'Predicted': predictions,
    'Error': np.asarray(y_test) - predictions
})

print('\nSample predictions:')
print(results.head(10).round(3))

print('\nModel comparison:')
print('Best CV RMSE:', round(-grid_search.best_score_, 3))
print('Test RMSE:', round(rmse, 3))


## Review notes

Inspect housing data, build train-only preprocessing pipelines, compare a baseline, and tune a random forest with cross-validation. First execution may download California housing through scikit-learn. No dataset or fitted model is bundled.

These are learning experiments, not production systems. Scores depend on the data split and package versions. Generated commentary in this publication is editorial context added during preparation, not a claim that it was part of the original class submission.